# 🧪 CausalCrisis Multimodal Pipeline


## §0 — Setup & Feature Extraction


In [ ]:
# 0. SETUP ENVIRONMENT & DATASET (ULTIMATE ROBUST VERSION)
# ⚡ Quiet install
!pip install -q open_clip_torch torch_geometric
!apt-get install -y aria2

import os, glob, subprocess
RAW_DATA_PATH = '/content/CrisisMMD_v2.0'
TAR_FILE = 'CrisisMMD_v2.0.tar.gz'

def setup_dataset():
    # 1. CLEANUP
    if os.path.exists(RAW_DATA_PATH):
        img_count = len(glob.glob(f'{RAW_DATA_PATH}/**/*.jpg', recursive=True))
        if img_count > 10000: # Dataset is likely complete (v2.0 has ~18k images)
            print(f'✅ Dataset already exists and is complete ({img_count} images).')
            return
    
    print('🧹 Cleaning up for a fresh, robust installation...')
    !rm -rf /content/CrisisMMD_v2.0*
    # Only wipe features if specifically requested or if data is corrupt
    # !rm -rf /content/extracted_features/*
    !rm -f /content/*.tar.gz*
    
    # 2. DOWNLOAD (QCRI Mirror - Official & Fast)
    url = 'https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz'
    print(f'🚀 Downloading from official mirror: {url}')
    !aria2c -x 16 -s 16 -k 1M --console-log-level=error {url}
    
    if not os.path.exists(TAR_FILE):
        print('⚠️ aria2 failed. Trying wget fallback...')
        !wget -q --show-progress {url}
        
    if not os.path.exists(TAR_FILE):
        print('❌ FATAL: Download failed. Please check internet connection or URL.')
        return
        
    # 3. EXTRACTION
    print('📦 Extracting TAR archive...')
    !tar -xf {TAR_FILE}
    !rm {TAR_FILE}
    
    # 4. HANDLE NESTED ZIP (Common in v2.0 for splits)
    zip_splits = glob.glob('/content/**/crisismmd_datasplit_all.zip', recursive=True)
    if zip_splits:
        print(f'📦 Found nested splits ZIP: {zip_splits[0]}. Unzipping...')
        !unzip -q {zip_splits[0]} -d {RAW_DATA_PATH}/
        
    # 5. NORMALIZE FOLDER STRUCTURE
    # If tar extracted into /content/CrisisMMD_v2.0/CrisisMMD_v2.0/
    nested_path = os.path.join(RAW_DATA_PATH, 'CrisisMMD_v2.0')
    if os.path.exists(nested_path):
        print('📦 Normalizing nested folders...')
        !mv {nested_path}/* {RAW_DATA_PATH}/ 2>/dev/null || true
        !rmdir {nested_path} 2>/dev/null || true
        
    # 6. VERIFY
    tsvs = glob.glob(f'{RAW_DATA_PATH}/**/*.tsv', recursive=True)
    imgs = glob.glob(f'{RAW_DATA_PATH}/**/*.jpg', recursive=True)
    print(f'✅ Setup complete!')
    print(f'📊 Found {len(tsvs)} TSV files.')
    print(f'🖼️ Found {len(imgs)} images.')
    if len(imgs) < 5000: print('⚠️ Warning: Image count looks low. Potential extraction issue.')

setup_dataset()


In [ ]:
# 1. IMPORT & SETUP
import os, glob, json, torch, random, numpy as np, pandas as pd, open_clip
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score, balanced_accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.manifold import TSNE
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'? Seed set to: {seed}')


set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RAW_DATA_DIR = '/content/CrisisMMD_v2.0'
DATASET_DIR = '/content/extracted_features'
os.makedirs(DATASET_DIR, exist_ok=True)
print(f'?? Device: {device}')


def build_knn_graph(features, k=4, sim_threshold=0.15, mutual=True, add_self_loop=True):
    """
    Build a more robust graph to reduce heterophily noise:
    - cosine top-k neighbors
    - similarity threshold pruning
    - optional mutual-kNN filtering
    - symmetric normalization
    """
    n = features.size(0)
    if n <= 1:
        return None

    feat = F.normalize(features, dim=1)
    sim = torch.matmul(feat, feat.T)
    sim.fill_diagonal_(-1.0)

    k_eff = min(k, n - 1)
    vals, idx = sim.topk(k_eff, dim=1)

    adj = torch.zeros(n, n, device=features.device)
    for i in range(n):
        keep = vals[i] >= sim_threshold
        if keep.any():
            adj[i, idx[i][keep]] = 1.0

    # Fallback: if graph too sparse, keep top-1 connection per node
    deg0 = (adj.sum(1) == 0)
    if deg0.any():
        for i in torch.where(deg0)[0].tolist():
            adj[i, idx[i, 0]] = 1.0

    # Mutual kNN for cleaner edges, fallback to union if over-pruned
    if mutual:
        adj_mutual = ((adj > 0) & (adj.T > 0)).float()
        if (adj_mutual.sum() > 0):
            adj = adj_mutual
        else:
            adj = ((adj + adj.T) > 0).float()

    # Ensure undirected graph
    adj = ((adj + adj.T) > 0).float()

    if add_self_loop:
        adj = adj + torch.eye(n, device=features.device)

    d = adj.sum(1)
    d_inv = torch.pow(d + 1e-9, -0.5)
    adj_norm = d_inv.unsqueeze(1) * adj * d_inv.unsqueeze(0)
    return adj_norm


In [ ]:
# 2. FAST FEATURE EXTRACTION (BATCHED + OOM FALLBACK)
import os, glob, json, hashlib, pandas as pd, numpy as np, torch
from PIL import Image
from tqdm import tqdm
import open_clip

# High-quality default as requested
CLIP_BACKBONE = 'ViT-L-14'
CLIP_PRETRAINED = 'openai'

TASK_FIXED_CLASSES = {
    'task2': [
        'affected_individuals',
        'infrastructure_and_utility_damage',
        'injured_or_dead_people',
        'missing_or_found_people',
        'not_humanitarian',
        'other_relevant_information',
        'rescue_volunteering_or_donation_effort',
        'vehicle_damage'
    ]
}


def _normalize_label_text(x):
    s = str(x).strip().lower()
    s = s.replace('&', 'and').replace('-', '_').replace(' ', '_')
    while '__' in s:
        s = s.replace('__', '_')
    return s


def _resolve_label_vocab(df, lbl_col, task, task_dir, data_split):
    task_key = str(task).lower()
    classes_path = f'{task_dir}/classes.npy'

    # Task2 keeps canonical order (needed by downstream 8->6 merge mapping).
    if task_key in TASK_FIXED_CLASSES:
        class_list = TASK_FIXED_CLASSES[task_key]
        return class_list, classes_path

    if data_split.lower() == 'train' or not os.path.exists(classes_path):
        class_list = sorted(df[lbl_col].dropna().astype(str).unique().tolist())
        return class_list, classes_path

    loaded = np.load(classes_path, allow_pickle=True)
    class_list = [str(x) for x in loaded.tolist()]
    return class_list, classes_path


def _select_metadata_tsv(all_tsvs, task, data_split):
    split_aliases = {
        'train': ['train'],
        'dev': ['dev', 'val', 'valid', 'validation'],
        'test': ['test']
    }
    task_keywords = {
        'task1': ['informative'],
        'task2': ['humanitarian'],
        'task3': ['damage']
    }

    split_keys = split_aliases.get(data_split.lower(), [data_split.lower()])
    task_keys = task_keywords.get(task.lower(), [task.lower()])

    candidates = []
    for tsv in sorted(all_tsvs):
        low = tsv.lower()
        base = os.path.basename(low)

        score = 0
        if any(k in low for k in task_keys):
            score += 5
        if any(k in base for k in split_keys):
            score += 5
        elif any(k in low for k in split_keys):
            score += 2

        other_splits = {'train', 'dev', 'val', 'valid', 'validation', 'test'} - set(split_keys)
        if any(k in base for k in other_splits):
            score -= 3

        if score > 0:
            candidates.append((score, len(tsv), tsv))

    if not candidates:
        return None

    candidates.sort(key=lambda x: (-x[0], x[1], x[2]))
    return candidates[0][2]


def _build_basename_index(root='/content/CrisisMMD_v2.0'):
    jpgs = glob.glob(f'{root}/**/*.jpg', recursive=True)
    index = {}
    for p in jpgs:
        b = os.path.basename(p)
        if b not in index:
            index[b] = p
    return index


def _resolve_image_path(rel_path, basename_index, root='/content/CrisisMMD_v2.0'):
    c1 = os.path.join(root, rel_path)
    if os.path.exists(c1):
        return c1

    c2 = os.path.join(root, rel_path.replace('data_image/', '', 1))
    if os.path.exists(c2):
        return c2

    return basename_index.get(os.path.basename(rel_path), None)


def _is_oom_error(err: Exception) -> bool:
    msg = str(err).lower()
    return ('out of memory' in msg) or ('cuda error: out of memory' in msg)


def _encode_clip_batch_with_fallback(img_tensors, texts, start_bs):
    """
    Encode a prepared list of tensors/texts with adaptive micro-batch fallback on OOM.
    Returns (img_feat_np, txt_feat_np, used_bs)
    """
    if len(img_tensors) == 0:
        return None, None, start_bs

    if device.type != 'cuda':
        # CPU path: single pass
        img_batch = torch.stack(img_tensors).to(device)
        txt_batch = tokenizer(texts).to(device)
        with torch.no_grad():
            i_f = clip_model.encode_image(img_batch)
            t_f = clip_model.encode_text(txt_batch)
            i_f = i_f / (i_f.norm(dim=-1, keepdim=True) + 1e-9)
            t_f = t_f / (t_f.norm(dim=-1, keepdim=True) + 1e-9)
        return i_f.cpu().numpy(), t_f.cpu().numpy(), start_bs

    # CUDA path with fallback ladder
    # requested ladder: 32 -> 24 -> 16 -> 8 -> 4
    ladder = [32, 24, 16, 8, 4]
    # pick nearest <= start_bs
    candidates = [b for b in ladder if b <= max(start_bs, 4)]
    if not candidates:
        candidates = [4]
    current_bs = max(candidates)

    all_i, all_t = [], []
    idx = 0
    used_bs = current_bs

    while idx < len(img_tensors):
        local_bs = min(current_bs, len(img_tensors) - idx)

        try:
            img_batch = torch.stack(img_tensors[idx:idx + local_bs]).to(device, non_blocking=True)
            txt_batch = tokenizer(texts[idx:idx + local_bs]).to(device, non_blocking=True)

            with torch.no_grad():
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    i_f = clip_model.encode_image(img_batch)
                    t_f = clip_model.encode_text(txt_batch)
                i_f = i_f / (i_f.norm(dim=-1, keepdim=True) + 1e-9)
                t_f = t_f / (t_f.norm(dim=-1, keepdim=True) + 1e-9)

            all_i.append(i_f.cpu())
            all_t.append(t_f.cpu())
            idx += local_bs

        except RuntimeError as e:
            if not _is_oom_error(e):
                raise

            torch.cuda.empty_cache()
            # reduce to next smaller ladder step
            smaller = [b for b in ladder if b < current_bs]
            if not smaller:
                raise
            current_bs = max(smaller)
            used_bs = min(used_bs, current_bs)
            print(f"?? OOM detected. Auto-reducing encode micro-batch to {current_bs} and retrying...")

    i_all = torch.cat(all_i, dim=0).numpy()
    t_all = torch.cat(all_t, dim=0).numpy()
    return i_all, t_all, used_bs


def extract_features(data_split='train', task='task1', require_ids=False):
    task_dir = os.path.join(DATASET_DIR, task)
    os.makedirs(task_dir, exist_ok=True)

    check_files = [
        f'{task_dir}/{data_split}_img.npy',
        f'{task_dir}/{data_split}_txt.npy',
        f'{task_dir}/{data_split}_labels.npy',
        f'{task_dir}/{data_split}_domains.npy',
        f'{task_dir}/classes.npy'
    ]
    if all(os.path.exists(f) for f in check_files):
        try:
            img_cached = np.load(f'{task_dir}/{data_split}_img.npy', mmap_mode='r')
            txt_cached = np.load(f'{task_dir}/{data_split}_txt.npy', mmap_mode='r')
            lbl_cached = np.load(f'{task_dir}/{data_split}_labels.npy')
            dom_cached = np.load(f'{task_dir}/{data_split}_domains.npy')
            cls_cached = np.load(f'{task_dir}/classes.npy', allow_pickle=True)

            ids_cached = np.load(f'{task_dir}/{data_split}_ids.npy', allow_pickle=True) if os.path.exists(f'{task_dir}/{data_split}_ids.npy') else None

            same_len = (
                len(img_cached) == len(txt_cached) == len(lbl_cached) == len(dom_cached)
                and len(lbl_cached) > 0
                and (ids_cached is None or len(ids_cached) == len(lbl_cached))
            )
            lbl_ok = (lbl_cached.min() >= 0) and (lbl_cached.max() < len(cls_cached))

            ids_ok = (not require_ids) or os.path.exists(f'{task_dir}/{data_split}_ids.npy')
            if same_len and lbl_ok and ids_ok:
                print(f"? Task {task.upper()} | Split {data_split.upper()}: Features found. Skipping extraction.")
                return True

            print(f"?? Cache check failed for {task}/{data_split}. Re-extracting this split...")
        except Exception as e:
            print(f"?? Cache read failed for {task}/{data_split}: {e}. Re-extracting...")

    print(f'\n?? --- PROCESSING TASK: {task.upper()} | SPLIT: {data_split.upper()} ---')

    all_tsvs = glob.glob('/content/**/*.tsv', recursive=True)
    target_tsv = _select_metadata_tsv(all_tsvs, task, data_split)
    if target_tsv is None:
        print(f'? ERROR: No TSV found for {task} {data_split}.')
        return False

    print(f'? Found metadata: {target_tsv}')
    df = pd.read_csv(target_tsv, sep='\t')

    global clip_model, preprocess, tokenizer, _basename_index, _clip_backbone_loaded
    if 'clip_model' not in globals() or globals().get('_clip_backbone_loaded') != CLIP_BACKBONE:
        print(f'?? Loading CLIP ({CLIP_BACKBONE}, {CLIP_PRETRAINED})...')
        clip_model, _, preprocess = open_clip.create_model_and_transforms(CLIP_BACKBONE, pretrained=CLIP_PRETRAINED)
        clip_model = clip_model.to(device).eval()
        tokenizer = open_clip.get_tokenizer(CLIP_BACKBONE)
        _basename_index = _build_basename_index('/content/CrisisMMD_v2.0')
        _clip_backbone_loaded = CLIP_BACKBONE
    elif '_basename_index' not in globals():
        _basename_index = _build_basename_index('/content/CrisisMMD_v2.0')

    if device.type != 'cuda':
        print('?? Running on CPU. Extraction will be very slow. Use GPU runtime for practical speed.')

    img_col = next((c for c in ['image_path', 'image', 'img_path'] if c in df.columns), df.columns[0])
    txt_col = next((c for c in ['tweet_text', 'text'] if c in df.columns), df.columns[1])
    lbl_col = next((c for c in ['label_text', 'label'] if c in df.columns), df.columns[2])
    dom_col = next((c for c in ['event_name', 'event'] if c in df.columns), None)

    df[lbl_col] = df[lbl_col].apply(_normalize_label_text)
    raw_unique = sorted(df[lbl_col].dropna().astype(str).unique().tolist())

    class_list, classes_path = _resolve_label_vocab(df, lbl_col, task, task_dir, data_split)
    if len(class_list) == 0:
        print(f'? ERROR: No label vocabulary found for {task}/{data_split}.')
        return False

    # Keep only known classes for this task/split.
    known = set(class_list)
    df = df[df[lbl_col].isin(known)].copy().reset_index(drop=True)
    if len(df) == 0:
        print(f'? ERROR: Empty split after filtering labels for {task}/{data_split}.')
        print(f'  - Observed labels (sample): {raw_unique[:10]}')
        print(f'  - Expected labels (sample): {class_list[:10]}')
        return False

    label_to_id = {name: idx for idx, name in enumerate(class_list)}
    encoded_labels = df[lbl_col].map(label_to_id).astype(np.int64).to_numpy()

    domain_vocab_path = f'{task_dir}/domains_vocab.json'
    if dom_col is None:
        domain_list = ['unknown']
    elif data_split.lower() == 'train' or not os.path.exists(domain_vocab_path):
        domain_list = sorted(df[dom_col].astype(str).unique().tolist())
        if 'unknown' not in domain_list:
            domain_list.append('unknown')
        with open(domain_vocab_path, 'w', encoding='utf-8') as f:
            json.dump(domain_list, f, ensure_ascii=False, indent=2)
    else:
        with open(domain_vocab_path, 'r', encoding='utf-8') as f:
            domain_list = json.load(f)
        if 'unknown' not in domain_list:
            domain_list.append('unknown')

    domain_map = {name: i for i, name in enumerate(domain_list)}
    unknown_domain_id = domain_map['unknown']

    img_feats, txt_feats, labels, domains, sample_ids = [], [], [], [], []
    current_batch_size = 32 if device.type == 'cuda' else 4
    skipped_missing, skipped_error = 0, 0

    print(f"?? Starting extraction batch size: {current_batch_size}")

    start_idx = 0
    pbar = tqdm(total=len(df), desc=f'Encoding {task}/{data_split}')

    while start_idx < len(df):
        batch_df = df.iloc[start_idx:start_idx + current_batch_size]

        img_tensors = []
        texts = []
        labels_batch = []
        domains_batch = []
        sample_ids_batch = []

        for i, row in batch_df.iterrows():
            abs_path = _resolve_image_path(str(row[img_col]), _basename_index)
            if abs_path is None:
                skipped_missing += 1
                continue

            try:
                image = Image.open(abs_path).convert('RGB')
                img_tensors.append(preprocess(image))
                texts.append(str(row[txt_col])[:200])
                labels_batch.append(int(encoded_labels[i]))
                domain_name = str(row[dom_col]) if dom_col is not None else 'unknown'
                domains_batch.append(domain_map.get(domain_name, unknown_domain_id))

                # Stable sample id used to align auxiliary-task labels for MTL.
                img_key = str(row[img_col]).strip().replace('\\', '/').lower()
                txt_key = str(row[txt_col]).strip().lower()
                txt_hash = hashlib.md5(txt_key.encode('utf-8')).hexdigest()[:10]
                sample_ids_batch.append(f"{img_key}::{txt_hash}")
            except Exception:
                skipped_error += 1
                continue

        if img_tensors:
            try:
                i_np, t_np, used_bs = _encode_clip_batch_with_fallback(img_tensors, texts, current_batch_size)
                if i_np is not None:
                    img_feats.append(i_np)
                    txt_feats.append(t_np)
                    labels.extend(labels_batch)
                    domains.extend(domains_batch)
                    sample_ids.extend(sample_ids_batch)

                if used_bs < current_batch_size:
                    current_batch_size = used_bs
                    print(f"?? New stable extraction batch size: {current_batch_size}")

            except RuntimeError as e:
                if _is_oom_error(e) and device.type == 'cuda':
                    torch.cuda.empty_cache()
                    current_batch_size = max(4, current_batch_size // 2)
                    print(f"?? Batch-level OOM. Reducing outer batch size to {current_batch_size} and continuing...")
                else:
                    raise

        processed = len(batch_df)
        start_idx += processed
        pbar.update(processed)

    pbar.close()

    if not img_feats:
        print(f'? ERROR: No features extracted for {task}/{data_split}.')
        return False

    np.save(f'{task_dir}/{data_split}_img.npy', np.vstack(img_feats))
    np.save(f'{task_dir}/{data_split}_txt.npy', np.vstack(txt_feats))
    np.save(f'{task_dir}/{data_split}_labels.npy', np.array(labels))
    np.save(f'{task_dir}/{data_split}_domains.npy', np.array(domains))
    np.save(f'{task_dir}/{data_split}_ids.npy', np.array(sample_ids, dtype=object))
    np.save(classes_path, np.array(class_list, dtype=object))

    if len(sample_ids) != len(labels):
        print(f'? WARNING: sample_ids length mismatch ({len(sample_ids)} vs {len(labels)}).')

    print(f'? Success: {len(labels)} samples processed | missing={skipped_missing} | errors={skipped_error}')
    return True


In [ ]:
# 🏃 RUN THIS to extract features (Wait ~5-10 mins for first run)
TASK_TO_RUN = 'task2' # Options: 'task1', 'task2', 'task3'

for split in ['train', 'dev', 'test']:
    success = extract_features(data_split=split, task=TASK_TO_RUN)
    if not success:
        print(f'❌ Failed to extract {split} features for {TASK_TO_RUN}')


## §1 — Core Components (Definitions)


### 🏗️ 1.1 Model Architectures


In [ ]:
## 🏗️ ARCHITECTURE: CROSS-MODAL ATTENTION FUSION
class CrossModalAttention(nn.Module):
    def __init__(self, embed_dim=512, num_heads=4, num_tokens=4):
        super().__init__()
        self.attn_scale = nn.Parameter(torch.ones(1) * 0.1)
        self.num_tokens = num_tokens
        self.token_dropout = nn.Dropout(0.1)
        self.t_attn_i = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.i_attn_t = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.fuse_layer = nn.Linear(embed_dim * 2, embed_dim)

    def _build_tokens(self, a, b):
        pair = 0.5 * (a + b)
        prod = a * b
        diff = a - b
        tokens = torch.stack([a, b, pair, prod, diff], dim=1)

        if self.num_tokens < tokens.size(1):
            tokens = tokens[:, :self.num_tokens, :]
        elif self.num_tokens > tokens.size(1):
            pad = tokens[:, -1:, :].expand(-1, self.num_tokens - tokens.size(1), -1)
            tokens = torch.cat([tokens, pad], dim=1)

        return self.token_dropout(tokens)

    def forward(self, img_feat, txt_feat):
        # Build short token sequences so cross-attention is non-trivial (SeqLen>1).
        img_tokens = self._build_tokens(img_feat, txt_feat)
        txt_tokens = self._build_tokens(txt_feat, img_feat)

        # Cross-modal interaction in both directions.
        attn_img, _ = self.t_attn_i(txt_tokens, img_tokens, img_tokens)
        attn_txt, _ = self.i_attn_t(img_tokens, txt_tokens, txt_tokens)

        # Aggregate attended token sets.
        out_img = self.norm1(img_tokens + attn_img).mean(dim=1)
        out_txt = self.norm2(txt_tokens + attn_txt).mean(dim=1)

        fused = self.fuse_layer(torch.cat([out_img, out_txt], dim=1))
        res = 0.5 * (img_feat + txt_feat)
        return (fused + res) * (1.0 + self.attn_scale)

class CausalCrisisV2Model(nn.Module):
    def __init__(self, num_classes=2, num_domains=4, hidden_dim=512):
        super().__init__()
        self.attn_scale = nn.Parameter(torch.ones(1) * 0.1)
        self.num_classes = num_classes
        self.visual_proj = nn.Linear(768, hidden_dim)
        self.text_proj = nn.Linear(768, hidden_dim)
        
        # NEW: Attention Fusion Module
        self.fusion = CrossModalAttention(embed_dim=hidden_dim)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
        # Disentanglement Heads (deeper + reconstruction)
        self.causal_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.spurious_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # 2-layer GNN Module
        self.gnn = nn.Linear(hidden_dim, hidden_dim)
        self.gnn2 = nn.Linear(hidden_dim, hidden_dim)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, img_feat, txt_feat, adj=None, backdoor_xs=None, **kwargs):
        # Step 0: Initial Projection
        v = self.visual_proj(img_feat)
        t = self.text_proj(txt_feat)
        
        # Step 1: Cross-Modal Attention Fusion (Interactive Reasoning)
        fused = self.fusion(v, t)
        fused = self.layer_norm(fused)
        
        # Step 2: Causal Disentanglement
        h_orig = fused
        xc = self.causal_head(h_orig)
        xs = self.spurious_head(h_orig)
        h_recon = self.decoder(torch.cat([xc, xs], dim=-1))

        # Step 3: Graph Context Enhancement (2-layer residual)
        if adj is not None:
            xg = torch.matmul(adj, xc)
            xg = F.gelu(self.gnn(xg))
            xg = torch.matmul(adj, xg)
            xg = self.gnn2(xg)
            xc = xc + 0.5 * xg
            
        # Step 4: Robust Inference (Logical Classifier)
        logits_gnn = self.classifier(xc)
        
        # Step 5: Backdoor Adjustment Fallback
        logits_ba = None
        if backdoor_xs is not None:
            if len(backdoor_xs.shape) == 2:
                N, D = backdoor_xs.shape
                xc_expanded = xc.unsqueeze(1).expand(-1, N, -1)
                intervened = xc_expanded + backdoor_xs.unsqueeze(0)
            else:
                B, N, D = backdoor_xs.shape
                xc_expanded = xc.unsqueeze(1).expand(-1, N, -1)
                intervened = xc_expanded + backdoor_xs
                
            logits_all = self.classifier(intervened.reshape(-1, D))
            probs_all = torch.softmax(logits_all.view(xc.shape[0], -1, self.num_classes), dim=-1)
            logits_ba = torch.log(probs_all.mean(dim=1) + 1e-9)
            
        return {'logits': logits_gnn, 'xc': xc, 'xs': xs, 'h_orig': h_orig, 'h_recon': h_recon, 'logits_ba': logits_ba}


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print('Causal V9 + V15++ model definitions loaded')


class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None


class ResidualMLPBlock(nn.Module):
    def __init__(self, dim=512, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return self.norm(x + self.net(x))


class MultiHeadCausalLinker(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim),
            nn.Sigmoid()
        )
        self.project = nn.Linear(dim, dim)
        nn.init.xavier_uniform_(self.project.weight, gain=0.01)

    def forward(self, xc, xs):
        combined = torch.cat([xc, xs], dim=-1)
        g = self.gate(combined)
        return xc + 0.5 * self.project(xs * g)


class BilinearResidualLinker(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.W = nn.Parameter(torch.empty(dim, dim))
        nn.init.xavier_uniform_(self.W)
        self.proj = nn.Linear(dim, dim)
        self.gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.LayerNorm(dim),
            nn.Sigmoid()
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, xc, xs):
        inter = torch.matmul(xc, self.W) * xs
        g = self.gate(torch.cat([xc, inter], dim=-1))
        return self.norm(xc + g * self.proj(inter))


class CausalCrisisV9Model(CausalCrisisV2Model):
    def __init__(self, num_classes=8, num_domains=11):
        super().__init__(num_classes, hidden_dim=512)
        self.domain_classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, num_domains)
        )
        self.causal_linker = MultiHeadCausalLinker(dim=512)

    def forward(self, img, txt, adj=None, alpha=1.0, backdoor_xs=None, labels=None, use_ba=False):
        out = super().forward(img, txt, adj)
        xc, xs = out['xc'], out['xs']

        rev_xs = GradReverse.apply(xs, alpha)
        domain_logits = self.domain_classifier(rev_xs)

        xc_refined = self.causal_linker(xc, xs)
        logits_main = self.classifier(xc_refined)

        logits_ba = None
        if use_ba and backdoor_xs is not None:
            B = xc.size(0)
            N_pool = backdoor_xs.size(0)
            N_ba = min(64, N_pool)
            idx = torch.randint(0, N_pool, (N_ba,), device=backdoor_xs.device)
            xs_samples = backdoor_xs[idx]
            xc_exp = xc.unsqueeze(1).expand(-1, N_ba, -1)
            xs_exp = xs_samples.unsqueeze(0).expand(B, -1, -1)
            xc_s = self.causal_linker(xc_exp.reshape(-1, 512), xs_exp.reshape(-1, 512))
            probs = torch.softmax(self.classifier(xc_s), dim=-1)
            logits_ba = torch.log(probs.view(B, N_ba, -1).mean(1) + 1e-9)

        return {
            'logits': logits_main,
            'logits_ba': logits_ba,
            'domain_logits': domain_logits,
            'xc': xc,
            'xs': xs,
            'h_orig': out.get('h_orig', None),
            'h_recon': out.get('h_recon', None)
        }


class CausalTransformerLinker(nn.Module):
    def __init__(self, dim=512, heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, dropout=0.1, batch_first=True)
        self.token_norm = nn.LayerNorm(dim)
        self.norm = nn.LayerNorm(dim)
        self.proj = nn.Linear(dim, dim)

    def _build_tokens(self, x_main, x_side):
        pair = 0.5 * (x_main + x_side)
        prod = x_main * x_side
        tokens = torch.stack([x_main, x_side, pair, prod], dim=1)
        return self.token_norm(tokens)

    def forward(self, xc, xs):
        # Use a compact token set to avoid degenerate SeqLen=1 attention.
        xc_q = self._build_tokens(xc, xs)
        xs_kv = self._build_tokens(xs, xc)
        context, _ = self.attn(xc_q, xs_kv, xs_kv)
        context = context.mean(dim=1)
        return self.norm(xc + self.proj(context))


class CausalCrisisV15Model(CausalCrisisV9Model):
    """V15++ architecture: deep disentangler + hybrid linker + gated graph + real MTL heads."""

    def __init__(self, num_classes=8, num_domains=11):
        super().__init__(num_classes, num_domains=num_domains)

        self.causal_head_plus = nn.Sequential(
            ResidualMLPBlock(512, dropout=0.2),
            ResidualMLPBlock(512, dropout=0.2),
            nn.Linear(512, 512),
            nn.LayerNorm(512)
        )
        self.spurious_head_plus = nn.Sequential(
            ResidualMLPBlock(512, dropout=0.2),
            ResidualMLPBlock(512, dropout=0.2),
            nn.Linear(512, 512),
            nn.LayerNorm(512)
        )

        self.causal_linker = CausalTransformerLinker(dim=512)
        self.bilinear_linker = BilinearResidualLinker(dim=512)
        self.fusion_gate = nn.Sequential(
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.Sigmoid()
        )
        self.refine_norm = nn.LayerNorm(512)

        self.graph_gate = nn.Sequential(
            nn.Linear(1024, 512),
            nn.LayerNorm(512),
            nn.Sigmoid()
        )
        self.graph_norm = nn.LayerNorm(512)

        self.minority_booster = nn.Sequential(
            nn.Linear(512, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 512)
        )

        # Real MTL heads (fix dead-code in trainer)
        self.head_task1 = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 2)
        )
        self.head_task3 = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 3)
        )

    def _refine_with_linkers(self, xc, xs):
        xc_attn = self.causal_linker(xc, xs)
        xc_bi = self.bilinear_linker(xc, xs)
        g = self.fusion_gate(torch.cat([xc_attn, xc_bi], dim=-1))
        xc_refined = self.refine_norm(g * xc_attn + (1.0 - g) * xc_bi)
        return xc_refined + 0.15 * self.minority_booster(xc_refined)

    def forward(self, img, txt, adj=None, alpha=1.0, backdoor_xs=None, labels=None, use_ba=False):
        base = super().forward(img, txt, adj=None, alpha=alpha, backdoor_xs=None, labels=labels, use_ba=False)
        h_orig = base['h_orig']

        xc = self.causal_head_plus(h_orig)
        xs = self.spurious_head_plus(h_orig)
        h_recon = self.decoder(torch.cat([xc, xs], dim=-1))

        rev_xs = GradReverse.apply(xs, alpha)
        domain_logits = self.domain_classifier(rev_xs)

        if adj is not None:
            xg = torch.matmul(adj, xc)
            xg = F.gelu(self.gnn(xg))
            xg = torch.matmul(adj, xg)
            xg = self.gnn2(xg)
            gg = self.graph_gate(torch.cat([xc, xg], dim=-1))
            xc = self.graph_norm(xc + gg * xg)

        xc_refined = self._refine_with_linkers(xc, xs)
        logits_main = self.classifier(xc_refined)
        logits_task1 = self.head_task1(xc_refined)
        logits_task3 = self.head_task3(xc_refined)

        logits_ba = None
        if use_ba and backdoor_xs is not None and backdoor_xs.shape[0] > 0:
            B, D = xc.shape
            N_pool = backdoor_xs.shape[0]
            N_ba = min(256, N_pool)
            idx = torch.randint(0, N_pool, (N_ba,), device=backdoor_xs.device)
            xs_samples = backdoor_xs[idx]

            xc_exp = xc.unsqueeze(1).expand(-1, N_ba, -1)
            xs_exp = xs_samples.unsqueeze(0).expand(B, -1, -1)
            xc_s = self._refine_with_linkers(xc_exp.reshape(-1, D), xs_exp.reshape(-1, D))
            probs = torch.softmax(self.classifier(xc_s).view(B, N_ba, -1), dim=-1)
            logits_ba = torch.log(probs.mean(1) + 1e-9)

        return {
            'logits': logits_main,
            'logits_ba': logits_ba,
            'domain_logits': domain_logits,
            'xc': xc,
            'xs': xs,
            'h_orig': h_orig,
            'h_recon': h_recon,
            'logits_task1': logits_task1,
            'logits_task3': logits_task3,
        }


### 🚂 1.2 Training Infrastructure


In [ ]:
class MemoryBankV9:
    def __init__(self, size=4000, dim=512, device='cuda'):
        self.device = device
        self.size = size
        self.bank = torch.zeros(size, dim, device=self.device).float()
        self.dom_bank = torch.full((size,), -1, device=self.device, dtype=torch.long)
        self.is_full = False
        self.ptr = 0

    def update(self, features, domains=None):
        b = features.size(0)
        if b <= 0:
            return

        feat = features.detach().float().to(self.device)
        dom = domains.detach().long().to(self.device) if domains is not None else None

        # If a batch is larger than bank size, keep the most recent entries.
        if b >= self.size:
            feat = feat[-self.size:]
            self.bank[:] = feat
            if dom is not None:
                self.dom_bank[:] = dom[-self.size:]
            self.ptr = 0
            self.is_full = True
            return

        space_left = self.size - self.ptr
        if b <= space_left:
            self.bank[self.ptr:self.ptr + b] = feat
            if dom is not None:
                self.dom_bank[self.ptr:self.ptr + b] = dom
            self.ptr += b
            if self.ptr == self.size:
                self.ptr = 0
                self.is_full = True
            return

        # Wrap-around write: tail then head.
        self.bank[self.ptr:] = feat[:space_left]
        remainder = b - space_left
        self.bank[:remainder] = feat[space_left:]
        if dom is not None:
            self.dom_bank[self.ptr:] = dom[:space_left]
            self.dom_bank[:remainder] = dom[space_left:]
        self.ptr = remainder
        self.is_full = True

    def _valid_size(self):
        return self.size if self.is_full else self.ptr

    def sample(self, n=50, stratified=False, num_domains=None):
        valid = self._valid_size()
        if valid <= 0:
            return self.bank[:0]

        if (not stratified) or (num_domains is None):
            idx = torch.randint(0, valid, (n,), device=self.device)
            return self.bank[idx]

        doms = self.dom_bank[:valid]
        valid_idx = torch.arange(valid, device=self.device)
        per_domain = max(1, n // max(1, num_domains))
        picked = []

        for d in range(num_domains):
            idx_d = valid_idx[doms == d]
            if idx_d.numel() == 0:
                continue
            if idx_d.numel() >= per_domain:
                choose = idx_d[torch.randperm(idx_d.numel(), device=self.device)[:per_domain]]
            else:
                rep = idx_d[torch.randint(0, idx_d.numel(), (per_domain,), device=self.device)]
                choose = rep
            picked.append(choose)

        if picked:
            idx = torch.cat(picked)
        else:
            idx = torch.randint(0, valid, (n,), device=self.device)

        if idx.numel() < n:
            extra = torch.randint(0, valid, (n - idx.numel(),), device=self.device)
            idx = torch.cat([idx, extra])
        elif idx.numel() > n:
            idx = idx[:n]

        return self.bank[idx]


def _pairwise_sqdist(x):
    x2 = (x * x).sum(-1, keepdim=True)
    dist = x2 + x2.T - 2.0 * (x @ x.T)
    return dist.clamp_min(0.0)


def _rbf_kernel(x, sigma=None):
    dist = _pairwise_sqdist(x)
    if sigma is None:
        vals = dist.detach().view(-1)
        vals = vals[vals > 0]
        sigma = torch.sqrt(vals.median() + 1e-6) if vals.numel() > 0 else torch.tensor(1.0, device=x.device)
    gamma = 1.0 / (2.0 * (sigma ** 2) + 1e-6)
    return torch.exp(-gamma * dist)


def hsic_loss(x, y):
    """Non-linear independence penalty (HSIC)."""
    n = x.size(0)
    if n < 4:
        return x.new_tensor(0.0)
    K = _rbf_kernel(x)
    L = _rbf_kernel(y)
    H = torch.eye(n, device=x.device) - (1.0 / n) * torch.ones((n, n), device=x.device)
    Kc = H @ K @ H
    Lc = H @ L @ H
    return (Kc * Lc).sum() / ((n - 1) ** 2 + 1e-9)


class FocalCrossEntropy(nn.Module):
    def __init__(self, class_weights=None, gamma=1.0):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma

    def forward(self, logits, targets):
        logp = F.log_softmax(logits, dim=-1)
        p = logp.exp()
        ce = F.nll_loss(logp, targets, weight=self.class_weights, reduction='none')
        pt = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        loss = ((1.0 - pt).clamp(min=1e-6) ** self.gamma) * ce
        return loss.mean()


class CausalTrainerV9:
    def __init__(
        self,
        model,
        optimizer,
        device,
        class_weights,
        priors=None,
        max_epochs=80,
        use_graph=True,
        graph_k=5,
        graph_warmup_epochs=None,
        num_domains=11,
    ):
        self.model = model
        self.optimizer = optimizer
        self.device = device
        self.max_epochs = max_epochs
        self.mem_bank = MemoryBankV9(dim=512, device=device)
        self.current_epoch = 0
        self.use_graph = use_graph
        self.graph_k = graph_k
        self.warmup_epochs = max(3, int(0.2 * self.max_epochs))
        self.graph_warmup_epochs = graph_warmup_epochs if graph_warmup_epochs is not None else self.warmup_epochs
        self.num_domains = num_domains

        if priors is not None:
            self.log_priors = torch.log(torch.tensor(priors).float() + 1e-9).to(device)
        else:
            self.log_priors = torch.log(torch.ones(len(class_weights)).float() / len(class_weights) + 1e-9).to(device)

        self.class_weights = torch.clamp(class_weights.float().to(device), min=0.6, max=1.8)
        self.criterion = nn.CrossEntropyLoss(weight=self.class_weights, label_smoothing=0.05)
        self.focal_criterion = FocalCrossEntropy(class_weights=self.class_weights, gamma=1.0)
        self.domain_criterion = nn.CrossEntropyLoss()

    def phase_for_epoch(self, epoch):
        return 'PHASE_1' if epoch <= self.warmup_epochs else 'PHASE_2_AND_3'

    def intervention_consistency(self, logits_raw, logits_ba):
        if logits_ba is None:
            return torch.tensor(0.0, device=self.device)
        return F.mse_loss(torch.softmax(logits_raw, dim=-1), torch.softmax(logits_ba, dim=-1))

    def _build_adj_from_causal(self, xc):
        if (not self.use_graph) or xc.size(0) <= 1:
            return None
        feat = F.normalize(xc, dim=1)
        return build_knn_graph(feat, k=self.graph_k, sim_threshold=0.12, mutual=True)

    def _get_causal_for_graph(self, img, txt, alpha=1.0):
        was_training = self.model.training
        self.model.eval()
        with torch.no_grad():
            pre = self.model(img, txt, adj=None, alpha=alpha, use_ba=False)
        if was_training:
            self.model.train()
        return pre['xc'].detach()

    def train_epoch(self, dataloader, epoch, phase=None):
        self.model.train()
        total_loss = 0.0
        self.current_epoch = epoch
        if phase is None:
            phase = self.phase_for_epoch(epoch)
        all_preds, all_labels = [], []
        grl_alpha = float(2.0 / (1.0 + np.exp(-10 * (epoch / self.max_epochs))) - 1.0)

        enable_graph = self.use_graph and (epoch > self.graph_warmup_epochs)

        for batch in tqdm(dataloader, desc=f"V15++ Train Ep {epoch}"):
            img = batch[0].to(self.device).float()
            txt = batch[1].to(self.device).float()
            labels = batch[2].to(self.device).long()
            dom_labels = batch[3].to(self.device).long()
            labels_task1 = batch[4].to(self.device).long() if len(batch) > 4 else None
            labels_task3 = batch[5].to(self.device).long() if len(batch) > 5 else None

            self.optimizer.zero_grad()
            use_ba = (phase != 'PHASE_1') and (self.mem_bank._valid_size() > 256)
            backdoor = self.mem_bank.sample(256, stratified=True, num_domains=self.num_domains) if use_ba else None

            adj = None
            if enable_graph:
                xc_for_graph = self._get_causal_for_graph(img, txt, alpha=grl_alpha)
                adj = self._build_adj_from_causal(xc_for_graph)

            out = self.model(img, txt, adj=adj, alpha=grl_alpha, use_ba=use_ba, backdoor_xs=backdoor)

            tau = 0.0 if phase == 'PHASE_1' else 0.35
            adj_logits = out['logits'] - tau * self.log_priors

            loss_ce = self.criterion(adj_logits, labels)
            loss_focal = self.focal_criterion(adj_logits, labels)
            w_focal = 0.0 if phase == 'PHASE_1' else 0.15
            loss_main = (1.0 - w_focal) * loss_ce + w_focal * loss_focal

            if labels_task1 is not None and labels_task3 is not None:
                loss_t1 = F.cross_entropy(out['logits_task1'], labels_task1)
                loss_t3 = F.cross_entropy(out['logits_task3'], labels_task3)
                loss_main = 0.65 * loss_main + 0.20 * loss_t1 + 0.15 * loss_t3

            loss_dom = self.domain_criterion(out['domain_logits'], dom_labels)

            xc_n = F.normalize(out['xc'], p=2, dim=1)
            xs_n = F.normalize(out['xs'], p=2, dim=1)
            loss_ortho = 2.0 * torch.mean(torch.abs(torch.sum(xc_n * xs_n, dim=1)))
            loss_hsic = hsic_loss(out['xc'], out['xs'])

            self.mem_bank.update(out['xs'], dom_labels)

            loss_recon = F.mse_loss(out['h_recon'], out['h_orig'])
            loss_int = self.intervention_consistency(out['logits'], out.get('logits_ba'))

            w_dom = 0.05 if phase == 'PHASE_1' else 0.08
            w_int = 0.0 if phase == 'PHASE_1' else 0.05
            w_hsic = 0.005 if phase == 'PHASE_1' else 0.02

            loss = loss_main + w_dom * loss_dom + 0.03 * loss_ortho + w_hsic * loss_hsic + 0.02 * loss_recon + w_int * loss_int

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            total_loss += loss.item()

            all_preds.extend(out['logits'].argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        all_preds = np.array(all_preds)
        if len(np.unique(all_preds)) <= 1:
            print('?? Collapse warning: model predicts a single class this epoch.')

        mean_loss = total_loss / max(len(dataloader), 1)
        return mean_loss, f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    @torch.no_grad()
    def evaluate(self, dataloader, return_details=False):
        self.model.eval()
        all_probs_raw, all_probs_ba, all_labels = [], [], []
        total_loss = 0.0
        use_ba = (self.current_epoch >= self.warmup_epochs)
        enable_graph = self.use_graph and (self.current_epoch > self.graph_warmup_epochs)

        for batch in dataloader:
            img = batch[0].to(self.device).float()
            txt = batch[1].to(self.device).float()
            labels = batch[2].to(self.device).long()

            adj = None
            if enable_graph:
                pre = self.model(img, txt, adj=None, alpha=1.0, use_ba=False)
                adj = self._build_adj_from_causal(pre['xc'])

            backdoor = self.mem_bank.sample(256, stratified=True, num_domains=self.num_domains) if self.mem_bank._valid_size() > 256 else None
            out = self.model(img, txt, adj=adj, use_ba=use_ba, backdoor_xs=backdoor)

            loss = self.criterion(out['logits'], labels)
            total_loss += loss.item()

            probs_raw = torch.softmax(out['logits'], dim=-1).cpu().numpy()
            probs_ba = torch.softmax(out['logits_ba'], dim=-1).cpu().numpy() if out['logits_ba'] is not None else probs_raw

            all_probs_raw.append(probs_raw)
            all_probs_ba.append(probs_ba)
            all_labels.extend(labels.cpu().numpy())

        all_probs_raw = np.vstack(all_probs_raw)
        all_probs_ba = np.vstack(all_probs_ba)
        all_labels = np.array(all_labels)

        preds_raw = np.argmax(all_probs_raw, axis=1)
        preds_ba = np.argmax(all_probs_ba, axis=1)

        result = {
            'loss': total_loss / max(len(dataloader), 1),
            'f1_raw': f1_score(all_labels, preds_raw, average='weighted', zero_division=0),
            'acc_raw': accuracy_score(all_labels, preds_raw),
            'macro_raw': f1_score(all_labels, preds_raw, average='macro', zero_division=0),
            'f1_ba': f1_score(all_labels, preds_ba, average='weighted', zero_division=0),
            'acc_ba': accuracy_score(all_labels, preds_ba),
            'macro_ba': f1_score(all_labels, preds_ba, average='macro', zero_division=0),
            'labels': all_labels,
            'preds_raw': preds_raw,
            'preds_ba': preds_ba,
            'probs_raw': all_probs_raw,
            'probs_ba': all_probs_ba,
            'use_ba': use_ba,
        }

        if return_details:
            return result

        if use_ba:
            print(f" (BA-F1: {result['f1_ba']:.4f}, BA-Macro: {result['macro_ba']:.4f})", end="")

        return result['loss'], result['f1_raw'], result['acc_raw'], result['f1_ba']


In [ ]:
## UTILITY: MEMORY BANK WARMUP (CRITICAL FOR BA EVALUATION)
def rebuild_memory_bank(trainer, dataloader):
    """
    Rebuild memory bank from train distribution for stable BA inference.
    """
    bank = getattr(trainer, 'memory_bank', getattr(trainer, 'mem_bank', None))
    if bank is None:
        print("?? Memory Bank not found in trainer. Skip warmup.")
        return

    print(f"?? Rebuilding Causal Memory Bank ({type(trainer).__name__}) from training distribution...")
    trainer.model.eval()

    if hasattr(bank, 'ptr'):
        bank.ptr = 0
        bank.is_full = False

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Bank Warmup"):
            img = batch[0].to(device).float()
            txt = batch[1].to(device).float()
            dom = batch[3].to(device).long() if len(batch) > 3 else None

            out = trainer.model(img, txt, adj=None, use_ba=False)
            xs = out.get('xs', None)
            if xs is not None:
                bank.update(xs, dom)

    print(f"? Bank Rebuilt. Size: {getattr(bank, 'ptr', 'N/A')}/{getattr(bank, 'size', 'N/A')}")


In [ ]:
## ??? UTILITY: SCIENTIFIC THRESHOLD OPTIMIZATION
def find_best_thresholds_scientificly(val_probs, val_labels, num_classes, num_points=500):
    """T?m ng??ng t?i ?u tr?n t?p VALIDATION (vectorized, kh?ng vi ph?m t?nh kh?ch quan)."""
    thresholds = np.full(num_classes, 0.5, dtype=np.float32)
    candidates = np.linspace(0.01, 0.99, num_points, dtype=np.float32)

    print("?? ?ang t?m ng??ng t?i ?u cho t?ng l?p ?? t?i ?a h?a F1...")
    for i in range(num_classes):
        y_true = (val_labels == i)
        probs = val_probs[:, i]

        preds = probs[:, None] >= candidates[None, :]
        tp = np.sum(preds & y_true[:, None], axis=0)
        fp = np.sum(preds & (~y_true[:, None]), axis=0)
        fn = np.sum((~preds) & y_true[:, None], axis=0)

        denom = (2 * tp + fp + fn).astype(np.float32)
        f1s = np.where(denom > 0, (2 * tp) / denom, 0.0)
        best_idx = int(np.argmax(f1s))
        thresholds[i] = float(candidates[best_idx])

        print(f"  - L?p {i}: Ng??ng t?t nh?t = {thresholds[i]:.2f} (F1 = {f1s[best_idx]:.4f})")

    return thresholds.tolist()


def apply_thresholds(probs, thresholds):
    """?p d?ng c?c ng??ng ?? t?m ???c ?? ??a ra d? ?o?n cu?i c?ng."""
    thr = np.clip(np.asarray(thresholds, dtype=np.float32), 1e-3, 1.0)
    adjusted_probs = probs / thr
    return np.argmax(adjusted_probs, axis=1)



### 📊 1.3 Data & Class Merging Logic


In [ ]:
# 2.5 CREATE DATALOADERS (ENHANCED WITH BALANCED SAMPLING)
from torch.utils.data import WeightedRandomSampler
def create_loader(split, task='task1', batch_size=64):
    task_dir = os.path.join(DATASET_DIR, task)
    path = f'{task_dir}/{split}_img.npy'
    if not os.path.exists(path):
        print(f'⚠️ Warning: {task}/{split} features missing.')
        return None
    
    img = torch.from_numpy(np.load(path, allow_pickle=True)).float()
    txt = torch.from_numpy(np.load(f'{task_dir}/{split}_txt.npy', allow_pickle=True)).float()
    lbl = torch.from_numpy(np.load(f'{task_dir}/{split}_labels.npy', allow_pickle=True)).long()
    dom = torch.from_numpy(np.load(f'{task_dir}/{split}_domains.npy', allow_pickle=True)).long()
    ds = TensorDataset(img, txt, lbl, dom)
    
    if split == 'train':
        # Mild re-balancing: avoid over-amplifying extreme minority classes.
        class_counts = torch.bincount(lbl, minlength=int(lbl.max().item()) + 1).float().clamp_min(1.0)
        weights = class_counts.pow(-0.35)
        weights = (weights / weights.mean()).clamp(0.7, 1.6)
        sample_weights = weights[lbl].double()
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
        return DataLoader(ds, batch_size=batch_size, sampler=sampler)
        
    return DataLoader(ds, batch_size=batch_size, shuffle=False)


In [ ]:
## ?? CLASS MERGING STRATEGY (Diff-Attention Paper Alignment)
# G?p 3 l?p thi?u s?: affected_individuals, injured_or_dead_people, missing_or_found_people
# ? Th?nh 1 l?p duy nh?t "affected_individuals" (gi?ng paper Diff-Attention)
# K?t qu?: Task 2 gi?m t? 8 l?p ? 6 l?p

import numpy as np, torch, os


def build_label_mapping(original_classes):
    """
    Original 8 classes (Task 2 CrisisMMD v2.0):
      0: affected_individuals
      1: infrastructure_and_utility_damage
      2: injured_or_dead_people
      3: missing_or_found_people
      4: not_humanitarian
      5: other_relevant_information
      6: rescue_volunteering_or_donation_effort
      7: vehicle_damage

    After merging (classes 2,3 -> 0 and 7 -> 1):
      0: affected_individuals (merged: affected + injured_dead + missing_found)
      1: infrastructure_and_utility_damage (merged with vehicle_damage)
      2: not_humanitarian
      3: other_relevant_information
      4: rescue_volunteering_or_donation_effort
    """
    mapping = {
        0: 0,
        1: 1,
        2: 0,
        3: 0,
        4: 2,
        5: 3,
        6: 4,
        7: 1,
    }

    new_class_names = np.array([
        'affected_individuals_merged',
        'infrastructure_and_utility_damage_merged',
        'not_humanitarian',
        'other_relevant_information',
        'rescue_volunteering_or_donation_effort'
    ])

    return mapping, new_class_names


def remap_labels(labels_array, mapping):
    """?p d?ng mapping l?n m?ng labels."""
    return np.array([mapping[int(l)] for l in labels_array])


def _load_split_ids(task, split):
    p = f'{DATASET_DIR}/{task}/{split}_ids.npy'
    if not os.path.exists(p):
        return None
    arr = np.load(p, allow_pickle=True)
    return arr.astype(str)


def _align_aux_labels_by_ids(base_ids, aux_ids, aux_labels):
    aux_map = {}
    for i, sid in enumerate(aux_ids):
        sid = str(sid)
        if sid not in aux_map:
            aux_map[sid] = i

    aligned = np.full(len(base_ids), -1, dtype=np.int64)
    missing = 0
    for i, sid in enumerate(base_ids):
        j = aux_map.get(str(sid), None)
        if j is None:
            missing += 1
        else:
            aligned[i] = int(aux_labels[j])
    return aligned, missing


def create_merged_loader(split, task='task2', batch_size=64, label_mapping=None, include_aux=False):
    """T?o DataLoader v?i labels ?? ???c g?p."""
    task_dir = os.path.join(DATASET_DIR, task)
    img_path = f'{task_dir}/{split}_img.npy'
    if not os.path.exists(img_path):
        print(f'?? Warning: {task}/{split} features missing.')
        return None

    img = torch.from_numpy(np.load(img_path, allow_pickle=True)).float()
    txt = torch.from_numpy(np.load(f'{task_dir}/{split}_txt.npy', allow_pickle=True)).float()
    lbl_raw = np.load(f'{task_dir}/{split}_labels.npy', allow_pickle=True)
    dom = torch.from_numpy(np.load(f'{task_dir}/{split}_domains.npy', allow_pickle=True)).long()

    if label_mapping is not None:
        lbl_remapped = remap_labels(lbl_raw, label_mapping)
    else:
        lbl_remapped = lbl_raw

    lbl = torch.from_numpy(lbl_remapped).long()
    tensors = [img, txt, lbl, dom]

    if include_aux:
        t1_path = f'{DATASET_DIR}/task1/{split}_labels.npy'
        t3_path = f'{DATASET_DIR}/task3/{split}_labels.npy'

        if os.path.exists(t1_path) and os.path.exists(t3_path):
            aux1_raw = np.load(t1_path, allow_pickle=True)
            aux3_raw = np.load(t3_path, allow_pickle=True)

            base_ids = _load_split_ids(task, split)
            ids_t1 = _load_split_ids('task1', split)
            ids_t3 = _load_split_ids('task3', split)

            aligned_ok = False
            strict_id_mismatch = False
            if base_ids is not None and ids_t1 is not None and ids_t3 is not None:
                aligned1, miss1 = _align_aux_labels_by_ids(base_ids, ids_t1, aux1_raw)
                aligned3, miss3 = _align_aux_labels_by_ids(base_ids, ids_t3, aux3_raw)

                if miss1 == 0 and miss3 == 0:
                    aux1 = torch.from_numpy(aligned1).long()
                    aux3 = torch.from_numpy(aligned3).long()
                    tensors.extend([aux1, aux3])
                    aligned_ok = True
                    print(f'{split}: loaded ID-aligned aux labels for Task1/Task3 (MTL).')
                else:
                    strict_id_mismatch = True
                    print(f'{split}: ID alignment missing (task1={miss1}, task3={miss3}), skip MTL labels for safety.')

            if (not aligned_ok) and (not strict_id_mismatch):
                aux1 = torch.from_numpy(aux1_raw).long()
                aux3 = torch.from_numpy(aux3_raw).long()
                if len(aux1) == len(lbl) and len(aux3) == len(lbl):
                    tensors.extend([aux1, aux3])
                    if base_ids is None or ids_t1 is None or ids_t3 is None:
                        print(f'{split}: aux labels loaded by index only (IDs missing).')
                    else:
                        print(f'{split}: aux labels loaded by index fallback (ID mismatch).')
                else:
                    print(f'{split}: aux label size mismatch, skip MTL labels.')
        else:
            print(f'{split}: Task1/Task3 labels missing, single-task mode.')

    ds = TensorDataset(*tensors)

    if split == 'train':
        class_counts = torch.bincount(lbl, minlength=int(lbl.max().item()) + 1).float().clamp_min(1.0)
        weights = class_counts.pow(-0.35)
        weights = (weights / weights.mean()).clamp(0.7, 1.6)
        sample_weights = weights[lbl].double()
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
        return DataLoader(ds, batch_size=batch_size, sampler=sampler)

    return DataLoader(ds, batch_size=batch_size, shuffle=False)


print("? Class Merging Strategy loaded. Ready to merge 8?5 classes.")



## ?2 ? V15 Unified Pipeline


In [ ]:
## V15+ UNIFIED PIPELINE
CURRENT_TASK = 'task2'
REQUIRE_IDS_FOR_MTL = False  # Set True if you want to force re-extraction for ID-aligned MTL.
print(f"Starting V15+ unified pipeline for {CURRENT_TASK.upper()}")


def _ensure_task_cache(task, splits=('train', 'dev', 'test'), require_ids=False):
    task_dir = f'{DATASET_DIR}/{task}'
    os.makedirs(task_dir, exist_ok=True)

    for split in splits:
        required = [
            f'{task_dir}/{split}_img.npy',
            f'{task_dir}/{split}_txt.npy',
            f'{task_dir}/{split}_labels.npy',
            f'{task_dir}/{split}_domains.npy',
            f'{task_dir}/classes.npy',
        ]
        ids_path = f'{task_dir}/{split}_ids.npy'

        missing_core = [p for p in required if not os.path.exists(p)]
        need_ids = require_ids and (not os.path.exists(ids_path))

        if missing_core or need_ids:
            print(f'Preparing {task}/{split}: missing_core={len(missing_core)} require_ids={need_ids}')
            ok = extract_features(data_split=split, task=task, require_ids=require_ids)
            if not ok:
                raise FileNotFoundError(f'Extraction failed for {task}/{split}. Check setup cells.')


_ensure_task_cache(CURRENT_TASK, require_ids=REQUIRE_IDS_FOR_MTL)
for aux_task in ['task1', 'task3']:
    _ensure_task_cache(aux_task, require_ids=REQUIRE_IDS_FOR_MTL)

if not REQUIRE_IDS_FOR_MTL:
    missing_id_info = []
    for t in [CURRENT_TASK, 'task1', 'task3']:
        for s in ['train', 'dev', 'test']:
            if not os.path.exists(f'{DATASET_DIR}/{t}/{s}_ids.npy'):
                missing_id_info.append(f'{t}/{s}')
    if missing_id_info:
        preview = ', '.join(missing_id_info[:5])
        print(f'ID files missing for {len(missing_id_info)} split(s): {preview}')
        print('MTL may fallback to index-based aux-label alignment on those splits.')

original_classes = np.load(f'{DATASET_DIR}/{CURRENT_TASK}/classes.npy', allow_pickle=True)
label_mapping, merged_classes = build_label_mapping(original_classes)
num_classes_merged = len(merged_classes)

print(f"Original classes: {len(original_classes)} -> Merged classes: {num_classes_merged}")
print(f"Merged class names: {list(merged_classes)}")

train_loader_m = create_merged_loader('train', task=CURRENT_TASK, label_mapping=label_mapping, include_aux=True)
val_loader_m = create_merged_loader('dev', task=CURRENT_TASK, label_mapping=label_mapping, include_aux=True)
test_loader_m = create_merged_loader('test', task=CURRENT_TASK, label_mapping=label_mapping, include_aux=False)

train_lbl_path = f'{DATASET_DIR}/{CURRENT_TASK}/train_labels.npy'
y_train_raw = np.load(train_lbl_path)
y_train_merged = remap_labels(y_train_raw, label_mapping)
counts_m = np.bincount(y_train_merged, minlength=num_classes_merged)
priors_m = counts_m / counts_m.sum()

weights_m = np.power(counts_m.astype(np.float32) + 1.0, -0.35)
weights_m = weights_m / weights_m.mean()
task2_weights_m = torch.tensor(np.clip(weights_m, 0.7, 1.6)).float().to(device)

print(f"Merged Priors: {priors_m}")
print(f"Merged Weights: {task2_weights_m}")

num_domains = 11
model_m = CausalCrisisV15Model(num_classes=num_classes_merged, num_domains=num_domains).to(device)

fast_tags = ('gnn', 'causal_linker', 'bilinear_linker', 'fusion_gate', 'graph_gate', 'head_plus', 'minority_booster')
optimizer_m = torch.optim.AdamW([
    {'params': [p for n, p in model_m.named_parameters() if not any(t in n for t in fast_tags)], 'lr': 4e-4},
    {'params': [p for n, p in model_m.named_parameters() if any(t in n for t in fast_tags)], 'lr': 1.2e-3}
], weight_decay=0.01)

scheduler_m = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_m, T_max=50, eta_min=1e-6)
trainer_m = CausalTrainerV9(
    model_m,
    optimizer_m,
    device,
    class_weights=task2_weights_m,
    priors=priors_m,
    max_epochs=50,
    use_graph=True,
    graph_k=5,
    graph_warmup_epochs=None,
    num_domains=num_domains,
)

print(f"V15+ pipeline ready. Training with {num_classes_merged} classes.")



In [ ]:
## V15 TRAINING (Early Stopping)
def _compute_checkpoint_score(val_result):
    # Unified checkpoint criterion shared across training blocks.
    return 0.7 * val_result['f1_raw'] + 0.3 * val_result['macro_raw']


best_score_m = -1.0
best_f1_m = 0.0
patience = 8
patience_counter = 0
num_epochs = 50

for epoch in range(1, num_epochs + 1):
    phase = trainer_m.phase_for_epoch(epoch)
    train_loss, _ = trainer_m.train_epoch(train_loader_m, epoch, phase=phase)
    val = trainer_m.evaluate(val_loader_m, return_details=True)
    scheduler_m.step()

    score = _compute_checkpoint_score(val)
    val_f1 = val['f1_raw']
    val_acc = val['acc_raw']
    val_macro = val['macro_raw']

    if score > best_score_m:
        best_score_m = score
        best_f1_m = val_f1
        patience_counter = 0
        torch.save(model_m.state_dict(), f'best_model_{CURRENT_TASK}_v15_merged.pt')
        print(f"Best model saved. Score: {score:.4f} | Val-F1: {val_f1:.4f} | Val-Macro: {val_macro:.4f}")
    else:
        patience_counter += 1

    print(
        f"V15 Ep {epoch:02d} | L:{train_loss:.3f} | "
        f"Score:{score:.4f} | Val-F1:{val_f1:.4f} | Val-Macro:{val_macro:.4f} | "
        f"Acc:{val_acc:.4f} | Pat:{patience_counter}/{patience}"
    )

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch}. Best Score: {best_score_m:.4f} | Best Val-F1: {best_f1_m:.4f}")
        break

print(f"Training complete. Best checkpoint score: {best_score_m:.4f} | Best validation F1: {best_f1_m:.4f}")



In [ ]:
## V15+ FINAL EVALUATION (LEAKAGE-FREE PROTOCOL)
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score

model_m.load_state_dict(torch.load(f'best_model_{CURRENT_TASK}_v15_merged.pt', map_location=device))

val_res = trainer_m.evaluate(val_loader_m, return_details=True)
test_res = trainer_m.evaluate(test_loader_m, return_details=True)

# Macro-first track selection to improve minority behavior
selected_track = 'ba' if val_res['macro_ba'] > val_res['macro_raw'] else 'raw'
val_probs = val_res[f'probs_{selected_track}']
val_labels = val_res['labels']
test_probs = test_res[f'probs_{selected_track}']
test_labels = test_res['labels']

val_preds_uncal = np.argmax(val_probs, axis=1)
val_f1_uncal = f1_score(val_labels, val_preds_uncal, average='weighted', zero_division=0)
val_macro_uncal = f1_score(val_labels, val_preds_uncal, average='macro', zero_division=0)

best_thresholds = find_best_thresholds_scientificly(val_probs, val_labels, num_classes_merged)
val_preds_cal = apply_thresholds(val_probs, best_thresholds)
val_f1_cal = f1_score(val_labels, val_preds_cal, average='weighted', zero_division=0)
val_macro_cal = f1_score(val_labels, val_preds_cal, average='macro', zero_division=0)

# Calibration decision from validation only (macro-first)
use_calibration = val_macro_cal > val_macro_uncal

test_preds_uncal = np.argmax(test_probs, axis=1)
test_preds_cal = apply_thresholds(test_probs, best_thresholds)
final_test_preds = test_preds_cal if use_calibration else test_preds_uncal

f1_test = f1_score(test_labels, final_test_preds, average='weighted', zero_division=0)
macro_test = f1_score(test_labels, final_test_preds, average='macro', zero_division=0)
bacc_test = balanced_accuracy_score(test_labels, final_test_preds)

print('=' * 60)
print('V15+ RESULTS ON TEST SET (LEAKAGE-FREE)')
print(f"Selected track (VAL macro): {selected_track.upper()}")
print(f"Use calibration (VAL macro): {use_calibration}")
print(f"Weighted F1 (TEST): {f1_test:.4f}")
print(f"Macro F1 (TEST): {macro_test:.4f}")
print(f"Balanced Acc (TEST): {bacc_test:.4f}")
print('=' * 60)

print('\nValidation selection details:')
print(f"VAL Weighted F1 uncalibrated: {val_f1_uncal:.4f}")
print(f"VAL Weighted F1 calibrated:   {val_f1_cal:.4f}")
print(f"VAL Macro F1 uncalibrated:    {val_macro_uncal:.4f}")
print(f"VAL Macro F1 calibrated:      {val_macro_cal:.4f}")

print('\n' + classification_report(test_labels, final_test_preds, target_names=merged_classes, zero_division=0))


## ?3 ? V15 Multi-Seed Experiment


In [ ]:
## V15 ARCHITECTURE NOTE
# Legacy architecture blocks were removed.
# V15 uses CausalCrisisV15Model defined in the core model cell above.
print('V15 architecture is active. Legacy blocks removed from this notebook.')


In [ ]:
## V15+ MULTI-SEED TRAINING
import copy

if '_compute_checkpoint_score' not in globals():
    def _compute_checkpoint_score(val_result):
        return 0.7 * val_result['f1_raw'] + 0.3 * val_result['macro_raw']

seeds = [42, 123, 2026]
all_models, all_best_f1 = [], []
total_epochs = 50

for seed_idx, seed in enumerate(seeds):
    print(f"\n{'='*60}\nV15+ Seed {seed_idx+1}/3 (seed={seed})\n{'='*60}")
    random.seed(seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    model_v15 = CausalCrisisV15Model(num_classes=num_classes_merged, num_domains=num_domains).to(device)
    fast_tags = ('gnn', 'causal_linker', 'bilinear_linker', 'fusion_gate', 'graph_gate', 'head_plus', 'minority_booster')
    optimizer_v15 = torch.optim.AdamW([
        {'params': [p for n, p in model_v15.named_parameters() if not any(t in n for t in fast_tags)], 'lr': 4e-4},
        {'params': [p for n, p in model_v15.named_parameters() if any(t in n for t in fast_tags)], 'lr': 1.2e-3}
    ], weight_decay=0.01)
    scheduler_v15 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_v15, T_max=total_epochs, eta_min=1e-6)

    trainer_v15 = CausalTrainerV9(
        model_v15,
        optimizer_v15,
        device=device,
        class_weights=task2_weights_m,
        priors=priors_m,
        max_epochs=total_epochs,
        use_graph=True,
        graph_k=5,
        graph_warmup_epochs=None,
        num_domains=num_domains,
    )

    best_score, best_f1, patience, counter = -1.0, 0.0, 8, 0
    for epoch in range(1, total_epochs + 1):
        phase = trainer_v15.phase_for_epoch(epoch)
        train_loss, _ = trainer_v15.train_epoch(train_loader_m, epoch, phase=phase)
        val = trainer_v15.evaluate(val_loader_m, return_details=True)
        scheduler_v15.step()

        score = _compute_checkpoint_score(val)
        if score > best_score:
            best_score = score
            best_f1 = val['f1_raw']
            counter = 0
            torch.save(model_v15.state_dict(), f'best_v15_seed{seed}.pt')
            print(
                f"Best ep {epoch:02d} | Score:{score:.4f} | "
                f"Val-F1(raw): {val['f1_raw']:.4f} | Val-Macro(raw): {val['macro_raw']:.4f} | "
                f"Val-F1(ba): {val['f1_ba']:.4f}"
            )
        else:
            counter += 1

        if counter >= patience:
            print(f'Early stop at ep {epoch}')
            break

    model_v15.load_state_dict(torch.load(f'best_v15_seed{seed}.pt', map_location=device))
    all_models.append(copy.deepcopy(model_v15))
    all_best_f1.append(best_f1)

print(f"\nV15+ complete. Mean F1(raw): {np.mean(all_best_f1):.4f} | Best: {max(all_best_f1):.4f}")


In [ ]:
## V15++ ENSEMBLE EVALUATION (WITH BA + VAL CALIBRATION)
from sklearn.metrics import classification_report, f1_score, accuracy_score, balanced_accuracy_score
from tqdm import tqdm


def build_backdoor_pool(model, loader, max_pool=3000, num_domains=None):
    model.eval()
    xs_all, dom_all = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc='Build BA pool', leave=False):
            img = batch[0].to(device).float()
            txt = batch[1].to(device).float()
            dom = batch[3].to(device).long() if len(batch) > 3 else None

            out = model(img, txt, adj=None, use_ba=False)
            xs = out['xs'].detach()
            xs_all.append(xs)
            if dom is not None:
                dom_all.append(dom)

            if sum(x.size(0) for x in xs_all) >= max_pool:
                break

    if not xs_all:
        return None

    pool = torch.cat(xs_all, dim=0)

    if (num_domains is None) or (not dom_all):
        if pool.size(0) > max_pool:
            idx = torch.randperm(pool.size(0), device=pool.device)[:max_pool]
            pool = pool[idx]
        return pool

    dom_vec = torch.cat(dom_all, dim=0)
    if len(dom_vec) != len(pool):
        if pool.size(0) > max_pool:
            idx = torch.randperm(pool.size(0), device=pool.device)[:max_pool]
            pool = pool[idx]
        return pool

    per_domain = max(1, max_pool // max(1, num_domains))
    picks = []
    full_idx = torch.arange(pool.size(0), device=pool.device)

    for d in range(num_domains):
        idx_d = full_idx[dom_vec == d]
        if idx_d.numel() == 0:
            continue
        if idx_d.numel() >= per_domain:
            picks.append(idx_d[torch.randperm(idx_d.numel(), device=pool.device)[:per_domain]])
        else:
            picks.append(idx_d[torch.randint(0, idx_d.numel(), (per_domain,), device=pool.device)])

    if picks:
        idx = torch.cat(picks)
    else:
        idx = full_idx

    if idx.numel() < max_pool:
        extra = full_idx[torch.randperm(full_idx.numel(), device=pool.device)[:(max_pool - idx.numel())]]
        idx = torch.cat([idx, extra])
    elif idx.numel() > max_pool:
        idx = idx[:max_pool]

    return pool[idx]


def ensemble_predict(models, loader, device, use_graph=True, graph_k=5, use_ba=False, backdoor_pools=None):
    for m in models:
        m.eval()

    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Ensemble'):
            img = batch[0].to(device).float()
            txt = batch[1].to(device).float()
            labels = batch[2].long()

            logits_list = []
            for mi, m in enumerate(models):
                adj = None
                if use_graph and img.size(0) > 1:
                    pre = m(img, txt, adj=None, use_ba=False)
                    feat = F.normalize(pre['xc'], dim=1)
                    adj = build_knn_graph(feat, k=graph_k, sim_threshold=0.12, mutual=True)

                pool = None
                if use_ba and backdoor_pools is not None:
                    pool = backdoor_pools[mi]

                out = m(img, txt, adj=adj, use_ba=use_ba, backdoor_xs=pool)
                if use_ba and (out.get('logits_ba') is not None):
                    logits_list.append(out['logits_ba'])
                else:
                    logits_list.append(out['logits'])

            probs = torch.stack([torch.softmax(lg, dim=1) for lg in logits_list]).mean(0)
            all_probs.append(probs.cpu())
            all_labels.extend(labels.numpy())

    return torch.cat(all_probs), np.array(all_labels)


print('Building BA pools for each seed model...')
backdoor_pools = [build_backdoor_pool(m, train_loader_m, max_pool=3000, num_domains=num_domains) for m in all_models]

# Test ensemble probabilities
probs_v15, labels_v15 = ensemble_predict(
    all_models,
    test_loader_m,
    device,
    use_graph=True,
    graph_k=5,
    use_ba=True,
    backdoor_pools=backdoor_pools,
)

# Validation ensemble probabilities for leakage-free calibration
probs_val_ens, labels_val_ens = ensemble_predict(
    all_models,
    val_loader_m,
    device,
    use_graph=True,
    graph_k=5,
    use_ba=True,
    backdoor_pools=backdoor_pools,
)

best_thresholds_ens = find_best_thresholds_scientificly(
    probs_val_ens.numpy(), labels_val_ens, num_classes_merged
)

preds_v15_raw = probs_v15.argmax(1).numpy()
preds_v15_cal = apply_thresholds(probs_v15.numpy(), best_thresholds_ens)

macro_raw = f1_score(labels_val_ens, probs_val_ens.argmax(1).numpy(), average='macro', zero_division=0)
macro_cal = f1_score(
    labels_val_ens,
    apply_thresholds(probs_val_ens.numpy(), best_thresholds_ens),
    average='macro',
    zero_division=0,
)

use_calibrated = macro_cal > macro_raw
preds_v15 = preds_v15_cal if use_calibrated else preds_v15_raw
print(f"Val macro (raw={macro_raw:.4f}, cal={macro_cal:.4f}) -> using {'calibrated' if use_calibrated else 'raw'} predictions")

print('\n' + '=' * 60 + f"\nV15++ FULL TEST ({num_classes_merged}-class, BA ensemble)\n" + '=' * 60)
f1_full_w = f1_score(labels_v15, preds_v15, average='weighted', zero_division=0)
f1_full_m = f1_score(labels_v15, preds_v15, average='macro', zero_division=0)
acc_full = accuracy_score(labels_v15, preds_v15)
bacc_full = balanced_accuracy_score(labels_v15, preds_v15)
print(f'Weighted F1: {f1_full_w:.4f}')
print(f'Macro F1:    {f1_full_m:.4f}')
print(f'Accuracy:    {acc_full:.4f}')
print(f'Bal. Acc:    {bacc_full:.4f}')

# Use merged class names dynamically to avoid mismatch when class-merge policy changes.
target_names = [str(x) for x in merged_classes]
print('\n' + classification_report(labels_v15, preds_v15, target_names=target_names, zero_division=0))


In [ ]:
## EXPORT PER-CLASS REPORT CSV (for auto-scorecard)
import os
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

if 'labels_v15' not in globals() or 'preds_v15' not in globals():
    raise RuntimeError('labels_v15/preds_v15 not found. Run ensemble evaluation cell first.')

class_labels = list(range(len(merged_classes)))
prec, rec, f1, sup = precision_recall_fscore_support(
    labels_v15,
    preds_v15,
    labels=class_labels,
    zero_division=0,
)

report_df = pd.DataFrame({
    'class': [str(c) for c in merged_classes],
    'precision': prec,
    'recall': rec,
    'f1-score': f1,
    'support': sup,
})

os.makedirs('results', exist_ok=True)
out_path = 'results/v15_class_report.csv'
report_df.to_csv(out_path, index=False)
print(f'Saved class report: {out_path}')
display(report_df)



## ?4 ? V15+ Ablation Study

So s?nh ??nh l??ng c?c th?nh ph?n ki?n tr?c ch?nh c?a V15+.


In [ ]:
## V15+ ABLATION SUITE (AUTO-RUN)
import time
import copy
from types import MethodType

if '_compute_checkpoint_score' not in globals():
    def _compute_checkpoint_score(val_result):
        return 0.7 * val_result['f1_raw'] + 0.3 * val_result['macro_raw']


def _set_seed_local(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class _CEProxy:
    """Used for w/o focal ablation: focal term behaves like CE."""
    def __init__(self, ce_loss):
        self.ce_loss = ce_loss

    def __call__(self, logits, targets):
        return self.ce_loss(logits, targets)


def _build_model_optimizer(num_classes, num_domains):
    model = CausalCrisisV15Model(num_classes=num_classes, num_domains=num_domains).to(device)
    fast_tags = ('gnn', 'causal_linker', 'bilinear_linker', 'fusion_gate', 'graph_gate', 'head_plus', 'minority_booster')
    optimizer = torch.optim.AdamW([
        {'params': [p for n, p in model.named_parameters() if not any(t in n for t in fast_tags)], 'lr': 4e-4},
        {'params': [p for n, p in model.named_parameters() if any(t in n for t in fast_tags)], 'lr': 1.2e-3}
    ], weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ABLATION_EPOCHS, eta_min=1e-6)
    return model, optimizer, scheduler


def _apply_ablation_hooks(model, trainer, cfg):
    # 1) w/o mutual-kNN (and threshold pruning)
    if cfg.get('no_mutual_knn', False):
        def _build_adj_relaxed(self, xc):
            if (not self.use_graph) or xc.size(0) <= 1:
                return None
            feat = F.normalize(xc, dim=1)
            return build_knn_graph(feat, k=self.graph_k, sim_threshold=0.0, mutual=False)
        trainer._build_adj_from_causal = MethodType(_build_adj_relaxed, trainer)

    # 2) w/o bilinear linker (attention-only linker)
    if cfg.get('no_bilinear', False):
        def _refine_attn_only(self, xc, xs):
            xc_attn = self.causal_linker(xc, xs)
            xc_ref = self.refine_norm(xc_attn)
            return xc_ref + 0.15 * self.minority_booster(xc_ref)
        model._refine_with_linkers = MethodType(_refine_attn_only, model)

    # 3) w/o focal loss (focal behaves as CE)
    if cfg.get('no_focal', False):
        trainer.focal_criterion = _CEProxy(trainer.criterion)


def _select_and_calibrate(val_res, test_res, num_classes, macro_selection=True, allow_track_switch=True):
    # Track selection from validation only
    if not allow_track_switch:
        selected_track = 'raw'
    elif macro_selection:
        selected_track = 'ba' if val_res['macro_ba'] > val_res['macro_raw'] else 'raw'
    else:
        selected_track = 'ba' if val_res['f1_ba'] > val_res['f1_raw'] else 'raw'

    val_probs = val_res[f'probs_{selected_track}']
    val_labels = val_res['labels']
    test_probs = test_res[f'probs_{selected_track}']
    test_labels = test_res['labels']

    val_preds_uncal = np.argmax(val_probs, axis=1)
    val_f1_uncal = f1_score(val_labels, val_preds_uncal, average='weighted', zero_division=0)
    val_macro_uncal = f1_score(val_labels, val_preds_uncal, average='macro', zero_division=0)

    thresholds = find_best_thresholds_scientificly(val_probs, val_labels, num_classes)
    val_preds_cal = apply_thresholds(val_probs, thresholds)
    val_f1_cal = f1_score(val_labels, val_preds_cal, average='weighted', zero_division=0)
    val_macro_cal = f1_score(val_labels, val_preds_cal, average='macro', zero_division=0)

    # Calibration decision from validation only
    use_cal = (val_macro_cal > val_macro_uncal) if macro_selection else (val_f1_cal > val_f1_uncal)

    test_preds = apply_thresholds(test_probs, thresholds) if use_cal else np.argmax(test_probs, axis=1)
    f1_w = f1_score(test_labels, test_preds, average='weighted', zero_division=0)
    f1_m = f1_score(test_labels, test_preds, average='macro', zero_division=0)
    bacc = balanced_accuracy_score(test_labels, test_preds)

    return {
        'selected_track': selected_track,
        'use_calibration': use_cal,
        'weighted_f1': float(f1_w),
        'macro_f1': float(f1_m),
        'balanced_acc': float(bacc),
    }


def run_single_ablation(cfg, seed):
    _set_seed_local(seed)

    model, optimizer, scheduler = _build_model_optimizer(num_classes_merged, num_domains)
    trainer = CausalTrainerV9(
        model,
        optimizer,
        device=device,
        class_weights=task2_weights_m,
        priors=priors_m,
        max_epochs=ABLATION_EPOCHS,
        use_graph=True,
        graph_k=5,
        graph_warmup_epochs=None,
        num_domains=num_domains,
    )
    _apply_ablation_hooks(model, trainer, cfg)

    best_state = None
    best_score = -1.0
    patience_ctr = 0

    for epoch in range(1, ABLATION_EPOCHS + 1):
        phase = trainer.phase_for_epoch(epoch)
        trainer.train_epoch(train_loader_m, epoch, phase=phase)
        val = trainer.evaluate(val_loader_m, return_details=True)
        scheduler.step()

        if cfg.get('weighted_selection', False):
            score = val['f1_raw']
        else:
            score = _compute_checkpoint_score(val)

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            patience_ctr = 0
        else:
            patience_ctr += 1

        if patience_ctr >= ABLATION_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    # Keep BA active in final validation/test evaluation
    trainer.current_epoch = max(trainer.current_epoch, 999)
    val_res = trainer.evaluate(val_loader_m, return_details=True)
    test_res = trainer.evaluate(test_loader_m, return_details=True)

    out = _select_and_calibrate(
        val_res,
        test_res,
        num_classes_merged,
        macro_selection=(not cfg.get('no_macro_selection', False)),
        allow_track_switch=(not cfg.get('force_raw_track', False))
    )
    out['seed'] = seed
    out['ablation'] = cfg['name']
    return out


def run_ablation_suite(configs, seeds):
    rows = []
    started = time.time()

    for cfg in configs:
        print('\n' + '=' * 80)
        print(f"Running ablation: {cfg['name']}")
        print('=' * 80)
        for seed in seeds:
            print(f"  - Seed {seed}")
            try:
                row = run_single_ablation(cfg, seed)
                rows.append(row)
                print(
                    f"    weighted_f1={row['weighted_f1']:.4f} | "
                    f"macro_f1={row['macro_f1']:.4f} | bacc={row['balanced_acc']:.4f}"
                )
            except Exception as e:
                print(f"    FAILED: {type(e).__name__}: {e}")

    if not rows:
        print('No successful ablation runs.')
        return None, None

    df = pd.DataFrame(rows)
    summary = df.groupby('ablation', as_index=False).agg(
        weighted_f1_mean=('weighted_f1', 'mean'),
        weighted_f1_std=('weighted_f1', 'std'),
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_std=('macro_f1', 'std'),
        bacc_mean=('balanced_acc', 'mean'),
        bacc_std=('balanced_acc', 'std'),
        n_runs=('seed', 'count')
    ).sort_values(by=['macro_f1_mean', 'weighted_f1_mean'], ascending=False)

    elapsed = (time.time() - started) / 60.0
    print(f"\nAblation suite done in {elapsed:.1f} minutes")
    return df, summary


# =========================
# User-tunable settings
# =========================
ABLATION_SEEDS = [42, 123]       # use [42,123,2026] for final report
ABLATION_EPOCHS = 25             # use 50 for full protocol
ABLATION_PATIENCE = 6            # use 8 for full protocol

ABLATION_CONFIGS = [
    {
        'name': 'full_v15_plus',
        'no_mutual_knn': False,
        'no_bilinear': False,
        'no_focal': False,
        'no_macro_selection': False,
        'weighted_selection': False,
    },
    {
        'name': 'w_o_mutual_knn',
        'no_mutual_knn': True,
        'no_bilinear': False,
        'no_focal': False,
        'no_macro_selection': False,
        'weighted_selection': False,
    },
    {
        'name': 'w_o_bilinear_linker',
        'no_mutual_knn': False,
        'no_bilinear': True,
        'no_focal': False,
        'no_macro_selection': False,
        'weighted_selection': False,
    },
    {
        'name': 'w_o_focal_loss',
        'no_mutual_knn': False,
        'no_bilinear': False,
        'no_focal': True,
        'no_macro_selection': False,
        'weighted_selection': False,
    },
    {
        'name': 'w_o_macro_selection',
        'no_mutual_knn': False,
        'no_bilinear': False,
        'no_focal': False,
        'no_macro_selection': True,
        'weighted_selection': True,
        'force_raw_track': True,
    },
]

ablation_raw_df, ablation_summary_df = run_ablation_suite(ABLATION_CONFIGS, ABLATION_SEEDS)

if ablation_summary_df is not None:
    print('\nAblation Summary (mean ? std):')
    display(ablation_summary_df)
    os.makedirs('results', exist_ok=True)
    ablation_summary_df.to_csv('results/v15_ablation_summary.csv', index=False)
    ablation_raw_df.to_csv('results/v15_ablation_raw.csv', index=False)
    print("Saved: results/v15_ablation_summary.csv and results/v15_ablation_raw.csv")


### Comparison with CrisisMMD Baselines (V15 run)

| Method | Weighted F1 | Balanced Acc | Source |
|:-------|:-----------:|:------------:|:-------|
| **Our Method (Causal V15)** | *Computed above* | *Computed above* | *This Study* |
| Differential Attention | ~0.83 | - | *Generic SOTA* |
| MMBT (Multimodal BERT) | ~0.80 | - | *Baseline* |

> Note: report only metrics generated by this V15 pipeline and matched protocol.
